# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SupreetOP/Flyrank/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

#Setting up of dataset before building up the vectors intial steps

In [4]:
!pip -q install duckdb

from google.colab import userdata
import duckdb

# Get Hugging Face token securely from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

# Connect to DuckDB
con = duckdb.connect()

# Authenticate DuckDB with Hugging Face
con.execute(
    f"""
    CREATE OR REPLACE SECRET hf_secret
    (TYPE huggingface, TOKEN '{HF_TOKEN}')
    """
)

# Hugging Face warehouse location
rel = "hf://datasets/FlyRank/internship-warehouse"

# Test access to the daily performance table
result = con.sql("""
    SELECT COUNT(*) AS row_count
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
""")

result

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────┐
│ row_count │
│   int64   │
├───────────┤
│  78835655 │
└───────────┘

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [5]:
feature_vector = con.sql("""
SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) AS gsc_impressions,
    SUM(gsc_clicks) AS gsc_clicks,
    AVG(gsc_avg_position) AS gsc_avg_position,
    SUM(ga4_pageviews) AS ga4_pageviews,
    SUM(ga4_engaged_sessions) AS ga4_engaged_sessions

FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)

WHERE month = '2026-03'

GROUP BY
    client_hash_id,
    content_hash_id
""").df()

feature_vector.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_engaged_sessions
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,1140.0,2.0,4.394234,0.0,0.0
1,client_73cda7b4e4f265ea,content_05597932fe4da067,57.0,0.0,2.714744,0.0,0.0
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,149.0,0.0,6.481453,4.0,0.0
3,client_73cda7b4e4f265ea,content_05434271b257bb68,1421.0,6.0,6.320337,12.0,0.0
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2770.0,16.0,4.459107,3.0,0.0


In [6]:
feature_vector.shape

(331437, 7)

In [7]:
feature_vector.columns.tolist()

['client_hash_id',
 'content_hash_id',
 'gsc_impressions',
 'gsc_clicks',
 'gsc_avg_position',
 'ga4_pageviews',
 'ga4_engaged_sessions']

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

## 2. Feature notes

| Feature                | Meaning                                                                                             | Missing-value handling                                                                                                                                                                    | Available when?                                                                             |
| ---------------------- | --------------------------------------------------------------------------------------------------- | ----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------- |
| `gsc_impressions`      | Number of Google Search Console impressions for the content during the March 2026 feature window.   | No missing values were observed in the March 2026 daily slice.                                                                                                                            | Available before the future prediction because it comes from the historical feature window. |
| `gsc_clicks`           | Number of Google Search Console clicks for the content during the March 2026 feature window.        | No missing values were observed in the March 2026 daily slice.                                                                                                                            | Available before the future prediction because it comes from the historical feature window. |
| `gsc_avg_position`     | Average Google Search Console search position for the content during the March 2026 feature window. | Missing for 6,230,317 of 9,841,378 daily rows. Missing values are retained as missing because zero is not a meaningful search position and missingness can indicate unavailable GSC data. | Available before the future prediction when GSC data is available.                          |
| `ga4_pageviews`        | Number of Google Analytics 4 pageviews for the content during the March 2026 feature window.        | Missing for 3,018,741 of 9,841,378 daily rows. Missingness is retained rather than automatically treating it as zero because it can represent unavailable GA4 data.                       | Available before the future prediction when GA4 data is available.                          |
| `ga4_engaged_sessions` | Number of GA4 engaged sessions for the content during the March 2026 feature window.                | Missing for 3,018,741 of 9,841,378 daily rows. Missingness is retained rather than automatically treating it as zero because it can represent unavailable GA4 data.                       | Available before the future prediction when GA4 data is available.                          |

**Categorical features:** None of the five selected features is categorical. `client_hash_id` and `content_hash_id` are identifiers/context fields, not model features.

**Decision-time rule:** All five features are derived from the March 2026 historical window. They must be frozen before the future outcome window is observed so that future information cannot leak into the prediction.


In [8]:
con.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_impressions IS NULL) AS missing_gsc_impressions,
    COUNT(*) FILTER (WHERE gsc_clicks IS NULL) AS missing_gsc_clicks,
    COUNT(*) FILTER (WHERE gsc_avg_position IS NULL) AS missing_gsc_avg_position,
    COUNT(*) FILTER (WHERE ga4_pageviews IS NULL) AS missing_ga4_pageviews,
    COUNT(*) FILTER (WHERE ga4_engaged_sessions IS NULL) AS missing_ga4_engaged_sessions
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
""")


┌────────────┬─────────────────────────┬────────────────────┬──────────────────────────┬───────────────────────┬──────────────────────────────┐
│ total_rows │ missing_gsc_impressions │ missing_gsc_clicks │ missing_gsc_avg_position │ missing_ga4_pageviews │ missing_ga4_engaged_sessions │
│   int64    │          int64          │       int64        │          int64           │         int64         │            int64             │
├────────────┼─────────────────────────┼────────────────────┼──────────────────────────┼───────────────────────┼──────────────────────────────┤
│    9841378 │                       0 │                  0 │                  6230317 │               3018741 │                      3018741 │
└────────────┴─────────────────────────┴────────────────────┴──────────────────────────┴───────────────────────┴──────────────────────────────┘

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [ ]:
feature_vector.columns.tolist()

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.